# 01 - Validation on the Curated Control Set

This notebook walks through `crypticip download-validation` and
`crypticip validate` on the 9-structure positive/negative control
set (5 cryptic IP-binding crystal/AF pairs + 4 PH-domain negatives).

**Heads up — Colab limitations:**
- Colab cannot run `apt-get install fpocket apbs pdb2pqr pymol` on
  free-tier runtimes reliably — the binaries are either unavailable
  in default repos or pull in large GUI dependencies.
- The most reliable Colab path is to install via `mamba`
  (condacolab) from `conda-forge` / `bioconda`. This takes
  ~5-10 minutes and ~3 GB of disk.
- The cleanest off-Colab path is the project `Dockerfile` (see
  `docs/reproducibility.md`).

Without these binaries, validation still runs but produces
`fpocket_status=missing` / `apbs_status=fallback` markers and the
result table will not reflect a real assessment. **Do not
interpret fallback-mode validation numbers as scientific results.**


## 1. Clone + install (run once per session)

In [ ]:
import os, subprocess, pathlib
REPO_URL = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
WORK = pathlib.Path('/content/cryptic-ip-pipeline')
if not WORK.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
!pip install -q -e .


## 2. (Optional) Install external binaries via condacolab + mamba

Skip this section if you want to see fallback behavior. Run it
(restarts the kernel!) if you want a real validation.

In [ ]:
INSTALL_EXTERNAL = False  # toggle to True for real run
if INSTALL_EXTERNAL:
    !pip install -q condacolab
    import condacolab; condacolab.install()  # NOTE: this restarts the kernel


In [ ]:
# Run *after* the kernel restarts (only if you set INSTALL_EXTERNAL=True above):
INSTALL_EXTERNAL = False
if INSTALL_EXTERNAL:
    !mamba install -y -c conda-forge -c bioconda \
        fpocket freesasa pdb2pqr apbs pymol-open-source python-freesasa
    import os; os.chdir('/content/cryptic-ip-pipeline')
    !pip install -q -e .


## 3. Check tool availability

In [ ]:
!crypticip check-env

## 4. Download the validation set

Pulls 5 PDB crystals from RCSB + 4 AlphaFold models from EBI.
Total: ~10 MB.

In [ ]:
!crypticip download-validation

## 5. Run validation

Iterates the controls, runs fpocket / FreeSASA / APBS, scores each
structure, and writes `results/reports/validation/validation_results.json`.

Expected wall-clock time (on full toolchain): 2-5 min.

In [ ]:
!crypticip validate || echo 'validate exited non-zero (fallback mode or gate failure)'

## 6. Generate the human-readable report

In [ ]:
!crypticip report --validation

## 7. Inspect ADAR2 diagnostics

In [ ]:
import json, pathlib, pandas as pd
vpath = pathlib.Path('results/reports/validation/validation_results.json')
if not vpath.exists():
    print('No validation_results.json — did `crypticip validate` succeed?')
else:
    report = json.loads(vpath.read_text())
    print('Gate:', report.get('gate'))
    print('Summary keys:', list((report.get('summary') or {}).keys()))
    rows = (report.get('structures') or [])
    if rows:
        df = pd.DataFrame(rows)
        adar = df[df['name'].str.contains('ADAR2', case=False, na=False)]
        if len(adar):
            display(adar.T)
        else:
            print('No ADAR2 rows; showing all:')
            display(df[['name','category','tier','score','fpocket_status','apbs_status']])


## 8. Troubleshooting

| Symptom | Fix |
|---|---|
| `fpocket: command not found` | Re-run section 2 with `INSTALL_EXTERNAL=True`, or use Docker (`docs/reproducibility.md`). |
| `apbs_status: fallback` everywhere | APBS missing; pipeline used a Coulomb fallback. Real numbers require APBS. |
| Download timeout | Re-run `crypticip download-validation`; it is idempotent. |
| `validate` exits 2 | Means the gate failed. Open `validation_results.json` to see which criterion. This may be a real failure or fallback-mode noise. |

**Reminder:** results from a partial toolchain are diagnostic only.
